<a href="https://colab.research.google.com/github/alexcho2823-del/ds2002-fa26/blob/main/03-pandas-cleaning/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


In [3]:
print(df.shape)

(315, 5)


### TODO 1 — drop duplicates

In [4]:
# TODO
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
log('dedupe', 'dropped exact duplicate rows', before - len(df))

[dedupe] dropped exact duplicate rows (15 row(s))


I need to remove the 15  duplicate rows that were ibjected in the data, so that no order gets double counted.

### TODO 2 — clean `price` -> float

In [5]:
# TODO
df['price'] = df['price'].str.replace('$', '', regex=False).astype(float)
log('price', 'stripped $ and cast to float', len(df))

[price] stripped $ and cast to float (300 row(s))


I need to Strip the $ from the price strings and then convert them to floats. This is so i can do math with them.

### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [6]:
# TODO
before = len(df)
df = df[df['qty'].notna() & (df['qty'] >= 1)]
df['qty'] = df['qty'].astype(int)
log('qty', 'dropped missing and negative qty', before - len(df))

[qty] dropped missing and negative qty (25 row(s))


I had to drop the rows where quantity is missing or negative, becayse you can't sell a negative or unknown number of items.

### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [7]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df['item'].value_counts())

ITEM_MAP = {
    'cheese burger': 'Cheeseburger',
    'rain poncho': 'Rain Poncho',
    'foam finger': 'Foam Finger',
}

old = df['item'].copy()
df['item'] = df['item'].replace(ITEM_MAP)
changed = (old != df['item']).sum()

log('item', f'collapsed {old.nunique()} spellings to {df["item"].nunique()} products', changed)
print(df['item'].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] collapsed 6 spellings to 3 products (126 row(s))
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


This cell lists the item spellings, maps the three lowercase or split variants onto their correct product names, and then logs that 126 rows were changed, which leaves exactly 3 products.



### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [8]:
# TODO
print(df['category'].value_counts())

CAT_MAP = {
    'food': 'Food',
    'rain-gear': 'RainGear',
    'Apparel': 'Merch',
}

old = df['category'].copy()
df['category'] = df['category'].replace(CAT_MAP)
changed = (old != df['category']).sum()

log('category', f'collapsed {old.nunique()} to {df["category"].nunique()}; merged Apparel into Merch since both are non-food souvenir goods', changed)
print(df['category'].value_counts())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] collapsed 6 to 3; merged Apparel into Merch since both are non-food souvenir goods (132 row(s))
category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64


This cell will merge the 6 category spellings into 3 by fixing `food` and `rain-gear` and folding Apparel into Merch as a business decision. It will then log that 132 rows have changed

### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [9]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item']) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
assert set(df['category']) == {'Food', 'Merch', 'RainGear'}
print('clean:', df.shape)

clean: (275, 5)


These asserst prove that the data is clean becasye it chgecks that there are no dupes, and that every quantity is at least 1, price is numeric, and item and category have only the expected canonical values

### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [10]:
# TODO
df['revenue'] = df['qty'] * df['price']

by_cat = df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)
print(by_cat.map('${:,.2f}'.format))
print(f"Total revenue: ${df['revenue'].sum():,.2f}")

category
Food        $1,656.00
Merch       $1,572.00
RainGear    $1,512.00
Name: revenue, dtype: object
Total revenue: $4,740.00


**What I would tell the vendor:** _..._ Stock slightly more Food, which leads at $1,656.00, though all 3 categories are within about 10% of each other, so no category is a clear winner.

### TODO 8 — read back your log

In [11]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,dedupe,dropped exact duplicate rows,15
1,price,stripped $ and cast to float,300
2,qty,dropped missing and negative qty,25
3,item,collapsed 6 spellings to 3 products,126
4,category,collapsed 6 to 3; merged Apparel into Merch si...,132


In [12]:
rng = np.random.default_rng(5)
raw_rows = []
for i in range(300):
    raw_rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
raw = pd.DataFrame(raw_rows)
raw = pd.concat([raw, raw.sample(15, random_state=1)])

def rev(d):
    p = d['price'].astype(str).str.replace('$', '', regex=False).astype(float)
    q = d['qty'].fillna(0)
    return round((q * p).sum(), 2)

s0 = raw
s1 = s0.drop_duplicates()
s2 = s1[s1['qty'].notna() & (s1['qty'] >= 1)]

print('Raw (with dupes):       ', rev(s0))
print('After dedupe:           ', rev(s1))
print('After qty filter:       ', rev(s2))
print('Alt: keep -1 as returns:', rev(s1[s1['qty'].notna()]))

Raw (with dupes):        4852.5
After dedupe:            4594.5
After qty filter:        4740.0
Alt: keep -1 as returns: 4594.5


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) Dropping the duplicates changed the revenue the most... from 4,852.50 dollars before to 4,594.50 dollars after (−258.00 dollars). The qty filter moved it by only plus 145.50 dollars (4,594.50 to 4,740.00).

b) I had dropped the 13 rows with qty = −1, but somoene defintley could have kept them as returns that would subtract from revenue. That would have reported 4,594.50 dollars instead of 4,740.00 dollars, which is 145 dollars lower. I had dropped them because the data had no returns flag or linked original order, so a −1 is  likely to be an entry error as a return